# 01c — Mock Conflict Database & Practice-Area Routing Schema

Creates the two supporting Delta tables for `02_agent`:
1. `lexpath_conflicts` — mock client DB for Tool 2 (Conflict Check) → CLEARED / CONFLICT_FLAG
2. `lexpath_routing_schema` — all 100 LEDGAR labels → firm practice areas, for Tool 3 (Case Routing)

In [0]:
# Configure Widgets for UC
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

CONFLICTS_TABLE = f"{CATALOG}.{SCHEMA}.lexpath_conflicts"
ROUTING_TABLE = f"{CATALOG}.{SCHEMA}.lexpath_routing_schema"

print(f"Conflicts table: {CONFLICTS_TABLE}")
print(f"Routing table:   {ROUTING_TABLE}")

Conflicts table: workspace.default.lexpath_conflicts
Routing table:   workspace.default.lexpath_routing_schema


In [0]:
# Mock Conflict Database
# Note: Atlas Manufacturing is deliberately listed as a client and an opposing party - used for conflict check test.
from pyspark.sql import functions as F

mock_matters = [
    # (matter_id, client_name, opposing_party, matter_type, assigned_attorney, status)
    ("M-1001", "Brightline Logistics LLC", "Hargrove Freight Co",     "Contract Dispute",      "D. Okafor",  "active"),
    ("M-1002", "Sandra Whitfield",         "Whitfield Family Trust",  "Estate Litigation",     "J. Tran",    "active"),
    ("M-1003", "NovaCore Technologies",    "Pinnacle Software Inc",   "IP Licensing",          "A. Reyes",   "active"),
    ("M-1004", "Meridian Property Group",  "City of Fairview",        "Zoning Appeal",         "L. Schmidt", "closed"),
    ("M-1005", "Tom Garrety",              "Atlas Manufacturing",     "Wrongful Termination",  "J. Tran",    "active"),
    ("M-1006", "Atlas Manufacturing",      "Union Local 482",         "Labor Negotiation",     "D. Okafor",  "closed"),
    ("M-1007", "Cobalt Ridge Ventures",    "Stonebrook Capital",      "Securities",            "A. Reyes",   "active"),
    ("M-1008", "Elena Marsh",              "Gregory Marsh",           "Divorce",               "L. Schmidt", "active"),
    ("M-1009", "Pinewood Medical Partners","HealthFirst Insurance",   "Coverage Dispute",      "J. Tran",    "active"),
    ("M-1010", "Drayton & Sons Roofing",   "Castellan HOA",           "Construction Defect",   "D. Okafor",  "closed"),
]

conflicts_df = spark.createDataFrame(
    mock_matters,
    schema="matter_id string, client_name string, opposing_party string, matter_type string, assigned_attorney string, status string"
).withColumn("created_at", F.current_timestamp())

(
    conflicts_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(CONFLICTS_TABLE)
)

display(spark.table(CONFLICTS_TABLE))

matter_id,client_name,opposing_party,matter_type,assigned_attorney,status,created_at
M-1001,Brightline Logistics LLC,Hargrove Freight Co,Contract Dispute,D. Okafor,active,2026-06-13T04:14:52.851Z
M-1002,Sandra Whitfield,Whitfield Family Trust,Estate Litigation,J. Tran,active,2026-06-13T04:14:52.851Z
M-1003,NovaCore Technologies,Pinnacle Software Inc,IP Licensing,A. Reyes,active,2026-06-13T04:14:52.851Z
M-1004,Meridian Property Group,City of Fairview,Zoning Appeal,L. Schmidt,closed,2026-06-13T04:14:52.851Z
M-1005,Tom Garrety,Atlas Manufacturing,Wrongful Termination,J. Tran,active,2026-06-13T04:14:52.851Z
M-1006,Atlas Manufacturing,Union Local 482,Labor Negotiation,D. Okafor,closed,2026-06-13T04:14:52.851Z
M-1007,Cobalt Ridge Ventures,Stonebrook Capital,Securities,A. Reyes,active,2026-06-13T04:14:52.851Z
M-1008,Elena Marsh,Gregory Marsh,Divorce,L. Schmidt,active,2026-06-13T04:14:52.851Z
M-1009,Pinewood Medical Partners,HealthFirst Insurance,Coverage Dispute,J. Tran,active,2026-06-13T04:14:52.851Z
M-1010,Drayton & Sons Roofing,Castellan HOA,Construction Defect,D. Okafor,closed,2026-06-13T04:14:52.851Z


In [0]:
# Test Conflict Check
def check_conflict(party_names: list[str]) -> dict:
    """Case-insensitive substring match against clients AND opposing parties."""
    df = spark.table(CONFLICTS_TABLE)
    hits = []
    for name in party_names:
        n = name.strip().lower()
        if not n:
            continue
        matched = df.filter(
            (F.lower(F.col("client_name")).contains(n)) |
            (F.lower(F.col("opposing_party")).contains(n))
        ).collect()
        hits.extend([(name, r.matter_id, r.client_name, r.opposing_party, r.status) for r in matched])
    return {
        "result": "CONFLICT_FLAG" if hits else "CLEARED",
        "matches": [
            {"query": h[0], "matter_id": h[1], "client": h[2], "opposing_party": h[3], "status": h[4]}
            for h in hits
        ],
    }

print(check_conflict(["Atlas Manufacturing"]))   # expect CONFLICT_FLAG (2 matters)
print(check_conflict(["Jane Doe"]))              # expect CLEARED

{'result': 'CONFLICT_FLAG', 'matches': [{'query': 'Atlas Manufacturing', 'matter_id': 'M-1005', 'client': 'Tom Garrety', 'opposing_party': 'Atlas Manufacturing', 'status': 'active'}, {'query': 'Atlas Manufacturing', 'matter_id': 'M-1006', 'client': 'Atlas Manufacturing', 'opposing_party': 'Union Local 482', 'status': 'closed'}]}
{'result': 'CLEARED', 'matches': []}


In [0]:
# Routing Schema
# Pull actual distinct labels from the dataset so every label gets mapped
labels = [
    r.category for r in
    spark.table("default.ledgar_lexglue")
        .select(F.col("category_label").getItem(0).alias("category"))
        .distinct()
        .collect()
]
print(f"{len(labels)} distinct LEDGAR categories found")

# Ordered keyword rules — FIRST match wins
RULES = [
    ("Intellectual Property", ["intellectual property", "licens", "trademark", "copyright", "patent"]),
    ("Employment",            ["employ", "salary", "compensation", "benefit", "vacation", "vesting",
                               "severance", "non-compet", "solicit", "erisa", "disabilit", "pension"]),
    ("Litigation",            ["arbitrat", "litigat", "indemnif", "remed", "release", "dispute",
                               "jurisdiction", "venue", "claims"]),
    ("Real Estate",           ["lease", "premises", "real property", "landlord", "tenant"]),
    ("Tax & Finance",         ["tax", "withhold", "interest", "payment", "fees", "expense",
                               "insurance", "loan", "indebtedness"]),
]
DEFAULT_AREA = "Corporate"

# Manual overrides for labels the keyword rules get wrong
OVERRIDES = {
    "Confidentiality": "Intellectual Property",
    "Governing Laws": "Litigation",
}

def assign_area(label: str) -> str:
    if label in OVERRIDES:
        return OVERRIDES[label]
    l = label.lower()
    for area, keywords in RULES:
        if any(k in l for k in keywords):
            return area
    return DEFAULT_AREA

routing_rows = [(label, assign_area(label)) for label in sorted(labels)]

routing_df = spark.createDataFrame(
    routing_rows, schema="category_label string, practice_area string"
).withColumn("created_at", F.current_timestamp())

(
    routing_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(ROUTING_TABLE)
)

100 distinct LEDGAR categories found


In [0]:
# Review the Mapping
display(
    spark.table(ROUTING_TABLE)
         .groupBy("practice_area")
         .agg(F.count("*").alias("label_count"),
              F.sort_array(F.collect_list("category_label")).alias("labels"))
         .orderBy(F.desc("label_count"))
)

practice_area,label_count,labels
Corporate,73,"List(Adjustments, Agreements, Amendments, Anti-Corruption Laws, Applicable Laws, Approvals, Assignments, Assigns, Authority, Authorizations, Binding Effects, Books, Brokers, Capitalization, Change In Control, Closings, Compliance With Laws, Consents, Construction, Cooperation, Costs, Counterparts, Death, Defined Terms, Definitions, Disclosures, Duties, Effective Dates, Effectiveness, Enforceability, Enforcements, Entire Agreements, Existence, Financial Statements, Forfeitures, Further Assurances, General, Headings, Indemnity, Integration, Interpretations, Liens, Miscellaneous, Modifications, No Conflicts, No Defaults, No Waivers, Non-Disparagement, Notices, Organizations, Participations, Positions, Powers, Publicity, Qualifications, Records, Representations, Sales, Sanctions, Severability, Solvency, Specific Performance, Subsidiaries, Successors, Survival, Terminations, Terms, Titles, Transactions With Affiliates, Use Of Proceeds, Waiver Of Jury Trials, Waivers, Warranties)"
Litigation,10,"List(Arbitration, Consent To Jurisdiction, Governing Laws, Indemnifications, Jurisdictions, Litigations, Releases, Remedies, Submission To Jurisdiction, Venues)"
Tax & Finance,8,"List(Expenses, Fees, Insurances, Interests, Payments, Tax Withholdings, Taxes, Withholdings)"
Employment,7,"List(Base Salary, Benefits, Disability, Employment, Erisa, Vacations, Vesting)"
Intellectual Property,2,"List(Confidentiality, Intellectual Property)"


In [0]:
# Completeness Check
n_labels = len(labels)
n_mapped = spark.table(ROUTING_TABLE).count()
assert n_labels == n_mapped, f"Mapping incomplete: {n_labels} labels vs {n_mapped} rows"
print(f"All {n_mapped} LEDGAR categories mapped to a practice area")

All 100 LEDGAR categories mapped to a practice area


## Summary
**`lexpath_conflicts`** — 10 fictional matters; Tool 2 does case-insensitive substring matching of intake party names against both `client_name` and `opposing_party`, returning CLEARED ✓ or CONFLICT_FLAG ✗ with matching matter details.
**`lexpath_routing_schema`** — every distinct LEDGAR category mapped to one of six practice areas via ordered keyword rules + manual overrides. Tool 3 joins the retrieval tool's predicted categories against this table to produce the routing JSON. Both tables are intentionally small and mock — per the proposal, conflict data is fictional to avoid any real client information.
